In [1]:
!pip install eyepop==3.12.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: cryptography
    Found existing installation: cryptography 49.0.0
    Uninstalling cryptography-49.0.0:
      Successfully uninstalled cryptography-49.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyopenssl 26.3.0 requires cryptography<50,>=49.0.0, but you have cryptography 46.0.7 which is incompatible.


In [2]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

Enter your Account UUID: a5184defa8e847248f589d35080efbfa
Enter your API KEY: ··········


In [3]:
NAMESPACE_PREFIX="datasciencealliance-org" # Add your namespace-prefix here

### Define Ability

In [4]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import InferenceComponent, Pop
import json


KITCHEN_SAFETY_COMPLIANCE_PROMPT = """
Analyze the commercial kitchen image and return structured kitchen safety compliance information.

Return only valid JSON. Do not include markdown, explanations, or extra commentary.

Use this exact JSON structure:

{
  "ppe_compliance": null,
  "hair_restraints": null,
  "gloves": null,
  "wet_floor_hazards": null,
  "blocked_exits": null,
  "unsafe_storage": null,
  "overall_safety_status": null,
  "visible_issues": [],
  "recommended_action": null,
  "visual_evidence": null
}

Instructions:

- "ppe_compliance" should summarize whether visible kitchen workers appear to be following basic PPE expectations. Use one of: "compliant", "partial_compliance", "non_compliant", "no_workers_visible", or "unknown".
Use "compliant" when visible workers appear to be wearing appropriate kitchen PPE such as hair restraints, gloves when handling food, aprons, or protective clothing.
Use "partial_compliance" when some PPE is visible but one or more expected items appear missing.
Use "non_compliant" when visible workers clearly lack important PPE.
Use "no_workers_visible" when no kitchen workers are visible.
Use "unknown" when the image is too unclear to assess.

- "hair_restraints" should describe whether visible food-handling workers appear to have hair properly restrained. Use one of: "compliant", "non_compliant", "not_applicable", or "unknown".
Use "compliant" when visible workers have hairnets, caps, hats, or tied-back/covered hair.
Use "non_compliant" when visible workers have uncovered loose hair near food prep or cooking areas.
Use "not_applicable" when no workers are visible.
Use "unknown" if heads or hair are not clearly visible.

- "gloves" should describe whether visible food-handling workers appear to be wearing gloves when handling ready-to-eat food or food preparation items. Use one of: "compliant", "non_compliant", "not_applicable", or "unknown".
Use "compliant" when visible workers handling food appear to be wearing gloves.
Use "non_compliant" when a visible worker is clearly handling food or food-contact items with bare hands.
Use "not_applicable" when no food handling is visible.
Use "unknown" when hands are not clearly visible.

- "wet_floor_hazards" should be true if there is a visible spill, puddle, wet floor area, mop water, reflective liquid, or slippery-looking floor hazard. Otherwise, use false. Use "unknown" if the floor is not visible or unclear.

- "blocked_exits" should be true if an exit door, emergency exit, doorway, or main passage appears blocked by boxes, carts, trash bins, equipment, stacked goods, or other obstacles. Otherwise, use false. Use "unknown" if exits or walkways are not visible.

- "unsafe_storage" should be true if items appear stored unsafely. This includes boxes stacked too high, items leaning, storage blocking walkways, heavy items stored above shoulder height, clutter near cooking equipment, supplies on the floor, or objects placed where they could fall, spill, or contaminate food. Otherwise, use false. Use "unknown" if storage areas are not visible.

- "overall_safety_status" should be one of: "safe", "minor_issues", "multiple_issues", "serious_issue", or "unknown".
Use "safe" when no clear safety issue is visible.
Use "minor_issues" when one small issue is visible.
Use "multiple_issues" when two or more safety issues are visible.
Use "serious_issue" when there is a blocked exit, major wet floor hazard, dangerous storage, or clearly unsafe condition.
Use "unknown" when the image is too unclear to assess.

- "visible_issues" should list the visible safety issues found in the image. Use short values such as "missing_hair_restraint", "no_gloves", "wet_floor", "blocked_exit", "unsafe_storage", "cluttered_walkway", or "none".

- "recommended_action" should briefly state what the kitchen team should do next, such as clean the spill, clear the exit, move unsafe storage, wear gloves, add hair restraints, or no immediate action needed.

- "visual_evidence" should briefly explain the visible clues supporting the assessment.

Important rules:
- Only assess what is visually observable in the image.
- Do not identify people or infer personal attributes.
- Do not guess violations that are not clearly visible.
- Do not rely on readable signs, captions, timestamps, camera overlays, or text labels.
- Ignore brand logos and restaurant names.
- If the image is ambiguous or too blurry, use "unknown" instead of guessing.

Return only the JSON object.
"""


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.describe.kitchen-safety-compliance",
        description="Analyze commercial kitchen images and return structured safety compliance details, including PPE compliance, hair restraints, gloves, wet floor hazards, blocked exits, and unsafe storage",
        worker_release="qwen3-instruct",
        text_prompt=KITCHEN_SAFETY_COMPLIANCE_PROMPT,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=350,
            image_size=640
        ),
        is_public=False
    )
]

### Create Ability

In [5]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

created ability 06a612d90f4d708a80006c094e2aa4cf with alias entries [AbilityAliasEntry(alias='datasciencealliance-org.describe.kitchen-safety-compliance', tag='1.0.1'), AbilityAliasEntry(alias='datasciencealliance-org.describe.kitchen-safety-compliance', tag='latest')]


### Evalulate on a Single Image

In [6]:
from pathlib import Path
import json
from eyepop import EyePopSdk
from eyepop.worker.worker_types import InferenceComponent, Pop


def extract_json_from_result(result):
    """
    Extract structured JSON from EyePop describe output.
    """
    for text_item in result.get("texts", []):
        text = text_item.get("text", "").strip()

        try:
            return json.loads(text)
        except json.JSONDecodeError:
            # Try to recover if the model wrapped JSON in extra text.
            start = text.find("{")
            end = text.rfind("}")

            if start != -1 and end != -1 and end > start:
                json_text = text[start:end + 1]

                try:
                    return json.loads(json_text)
                except json.JSONDecodeError:
                    pass

    return None


pop = Pop(components=[
    InferenceComponent(
        ability=f"{NAMESPACE_PREFIX}.describe.kitchen-safety-compliance:latest"
    )
])


# Replace this with your actual image path.
input_path = Path("/content/sample_kitchen_safety_image.jpg")


raw_results = []

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
    endpoint.set_pop(pop)

    job = endpoint.upload(input_path)

    while result := job.predict():
        raw_results.append(result)


print("=== RAW EYEPOP RESULT ===")
print(json.dumps(raw_results[:1], indent=2))


print("\n=== KITCHEN SAFETY COMPLIANCE OUTPUT ===")

if raw_results:
    structured_output = extract_json_from_result(raw_results[0])

    if structured_output:
        print(json.dumps(structured_output, indent=2))
    else:
        print("No valid JSON detected.")
else:
    print("No result returned.")

=== RAW EYEPOP RESULT ===
[
  {
    "compute_units": 0.40700000000000003,
    "seconds": 0,
    "source_height": 559,
    "source_id": "50c6bade-860f-11f1-a66d-fee1f62d0fc5",
    "source_width": 1024,
    "system_timestamp": 1784753587002501000,
    "texts": [
      {
        "id": 1,
        "text": "{\n  \"ppe_compliance\": \"compliant\",\n  \"hair_restraints\": \"compliant\",\n  \"gloves\": \"non_compliant\",\n  \"wet_floor_hazards\": true,\n  \"blocked_exits\": false,\n  \"unsafe_storage\": true,\n  \"overall_safety_status\": \"multiple_issues\",\n  \"visible_issues\": [\n    \"no_gloves\",\n    \"wet_floor\",\n    \"unsafe_storage\"\n  ],\n  \"recommended_action\": \"Clean the wet floor immediately to prevent slips and falls, require staff to wear gloves when handling ready-to-eat food, and reorganize storage to prevent boxes from falling and to clear walkways.\",\n  \"visual_evidence\": \"Two workers are visible; both are wearing hairnets and aprons but are handling food with bar